# 📸 Google Photos Storage Optimizer (Cloud Colab Runner)

Run this notebook directly in Google's cloud using Google Colab to **find burst duplicates, near-identical photos, and oversized files without downloading anything to your home computer**.

---

### How the Zero-Bandwidth Cloud Flow Works:
1. Go to [Google Takeout](https://takeout.google.com) and select **Google Photos** (you can select specific years or albums).
2. For **Destination**, choose **"Add to Drive"**. Google transfers the zip directly to Google Drive inside Google's datacenter — **0 MB downloaded to your computer**.
3. Run this Colab notebook. It mounts the Takeout zip, unzips it to Colab's cloud SSD, and scans with perceptual difference hashing (`dHash`) and Laplacian focus scoring.
4. It outputs an interactive **HTML Cleanup Report** with direct 1-click links to Google Photos so you can delete redundant burst frames and reclaim storage.
5. When finished, delete the Takeout zip from Drive to free that temporary cloud space.

### Step 1: Install Required Libraries

In [ ]:
!pip install -q Pillow imagehash tqdm pandas

### Step 2: Connect to Google Drive
Mount your Google Drive to access the Takeout archive exported to Drive.

In [ ]:
from google.colab import drive
import os, glob, zipfile, json, shutil
from pathlib import Path

drive.mount('/content/drive')

# Search for Takeout archives in Google Drive
drive_path = '/content/drive/MyDrive'
takeout_zips = glob.glob(f'{drive_path}/**/takeout*.zip', recursive=True) + glob.glob(f'{drive_path}/takeout*.zip')

print(f"Found {len(takeout_zips)} Takeout zip(s):")
for i, z in enumerate(takeout_zips):
    print(f"  [{i}] {z} ({os.path.getsize(z) / (1024*1024):.1f} MB)")

### Step 3: Extract Photos to Fast Colab Cloud Scratchpad
Extract the photos to `/content/photos_workdir` on Colab's high-speed NVMe scratch drive.

In [ ]:
WORKDIR = '/content/photos_workdir'
os.makedirs(WORKDIR, exist_ok=True)

if takeout_zips:
    for zip_path in takeout_zips:
        print(f"Extracting {os.path.basename(zip_path)} to fast cloud disk...")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(WORKDIR)
    print("Extraction complete!")
else:
    print("No takeout zip found in Google Drive root. If you manually placed photos in a folder, set WORKDIR to that path.")

### Step 4: Run Perceptual Hashing & Sharpness Scoring
Scans all images using:
- **dHash (64-bit difference hashing)** to detect near-duplicates and bursts regardless of minor lighting changes.
- **Laplacian Variance Focus Scoring** to rank image sharpness and pick the best shot in every burst.

In [ ]:
import cv2
import numpy as np
from PIL import Image, ExifTags
import imagehash
from tqdm.notebook import tqdm
import datetime

SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.heic'}
photo_entries = []

def get_image_date(file_path):
    # Check for accompanying Takeout JSON metadata file
    json_path = file_path + '.json'
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
                ts = int(meta.get('photoTakenTime', {}).get('timestamp', 0))
                if ts > 0:
                    return datetime.datetime.fromtimestamp(ts)
        except Exception:
            pass
    
    # Fallback to EXIF
    try:
        with Image.open(file_path) as img:
            exif = img._getexif()
            if exif:
                for tag_id, value in exif.items():
                    tag = ExifTags.TAGS.get(tag_id, tag_id)
                    if tag == 'DateTimeOriginal':
                        return datetime.datetime.strptime(value, '%Y:%m:%d %H:%M:%S')
    except Exception:
        pass
    
    # Fallback to file mtime
    return datetime.datetime.fromtimestamp(os.path.getmtime(file_path))

def calculate_sharpness(img_path):
    try:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return 50.0
        # Laplacian variance measures edge frequency and focus
        variance = cv2.Laplacian(img, cv2.CV_64F).var()
        return float(min(100.0, variance / 10.0))
    except Exception:
        return 50.0

# Find all photo files
all_files = []
for root, dirs, files in os.walk(WORKDIR):
    for file in files:
        if Path(file).suffix.lower() in SUPPORTED_EXTS:
            all_files.append(os.path.join(root, file))

print(f"Analyzing {len(all_files)} photos in cloud memory...")

records = []
for path in tqdm(all_files):
    try:
        size_bytes = os.path.getsize(path)
        with Image.open(path) as img:
            width, height = img.size
            dhash = imagehash.dhash(img)
            
        sharpness = calculate_sharpness(path)
        taken_time = get_image_date(path)
        mp = (width * height) / 1_000_000.0
        
        # Quality score
        score = (sharpness * 1.5) + (min(24.0, mp) * 2.0)
        
        records.append({
            'path': path,
            'filename': os.path.basename(path),
            'size_mb': size_bytes / (1024 * 1024),
            'width': width,
            'height': height,
            'mp': round(mp, 1),
            'dhash': dhash,
            'sharpness': round(sharpness, 1),
            'score': round(score, 1),
            'taken_at': taken_time
        })
    except Exception as e:
        continue

print(f"Successfully indexed {len(records)} images!")

### Step 5: Detect Burst Clusters & Duplicates
Groups photos taken within 90 seconds of each other with similar perceptual visual hashes.

In [ ]:
# Sort chronologically
records.sort(key=lambda x: x['taken_at'])

clusters = []
assigned = set()

for i in range(len(records)):
    if i in assigned:
        continue
        
    group = [i]
    for j in range(i + 1, len(records)):
        if j in assigned:
            continue
            
        # Check time window (within 90 seconds for burst shots)
        delta_sec = abs((records[j]['taken_at'] - records[i]['taken_at']).total_seconds())
        if delta_sec > 120:  # Burst window
            break
            
        # Check Hamming distance of dHash (0-64; <= 8 is very similar)
        dist = records[i]['dhash'] - records[j]['dhash']
        if dist <= 8:
            group.append(j)
            assigned.add(j)
            
    if len(group) > 1:
        assigned.add(i)
        clusters.append([records[idx] for idx in group])

print(f"Found {len(clusters)} burst/duplicate clusters!")

# Mark best shot in each cluster
redundant_files = []
total_redundant_mb = 0.0

for cl in clusters:
    cl.sort(key=lambda x: x['score'], reverse=True)
    best = cl[0]
    redundant = cl[1:]
    for item in redundant:
        redundant_files.append({
            'filename': item['filename'],
            'date': item['taken_at'].strftime('%Y-%m-%d'),
            'size_mb': item['size_mb'],
            'reason': f"Burst duplicate of {best['filename']} (Best sharpness: {best['sharpness']} vs {item['sharpness']})",
            'path': item['path']
        })
        total_redundant_mb += item['size_mb']

print(f"\nSummary of Storage Optimization:")
print(f"- Redundant Burst/Duplicate Photos: {len(redundant_files)}")
print(f"- Potential Storage Space Saved: {total_redundant_mb:.1f} MB ({total_redundant_mb/1024:.2f} GB)")

### Step 6: Generate 1-Click Interactive Cleanup Report
Creates an interactive HTML file with direct search links to Google Photos (`https://photos.google.com/search/...`) by date and filename so you can delete them in a few clicks.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

df_redundant = pd.DataFrame(redundant_files)
df_redundant.to_csv('/content/redundant_photos_to_delete.csv', index=False)

html_content = f"""
<html>
<head>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 20px; background: #f8fafc; color: #1e293b; }}
  h1 {{ color: #0f172a; }}
  .banner {{ background: #e0f2fe; border: 1px solid #bae6fd; padding: 15px; border-radius: 8px; margin-bottom: 20px; }}
  table {{ width: 100%; border-collapse: collapse; background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
  th, td {{ padding: 12px 16px; text-align: left; border-bottom: 1px solid #e2e8f0; font-size: 14px; }}
  th {{ background: #f1f5f9; font-weight: 600; color: #475569; }}
  .badge {{ background: #fee2e2; color: #991b1b; padding: 4px 8px; border-radius: 4px; font-weight: 600; font-size: 12px; }}
  .btn {{ background: #2563eb; color: white; text-decoration: none; padding: 6px 12px; border-radius: 4px; font-size: 13px; font-weight: 500; }}
  .btn:hover {{ background: #1d4ed8; }}
</style>
</head>
<body>
  <h1>Google Photos Deletion & Storage Optimizer Report</h1>
  <div class='banner'>
    <strong>Reclaimable Cloud Space:</strong> {total_redundant_mb / 1024:.2f} GB ({len(redundant_files)} photos)<br>
    Click <strong>Find in Google Photos</strong> on each date to jump straight to those photos on photos.google.com and trash the burst duplicates.
  </div>
  <table>
    <thead>
      <tr>
        <th>Date</th>
        <th>File Name</th>
        <th>Size</th>
        <th>Status</th>
        <th>Action</th>
      </tr>
    </thead>
    <tbody>
"""

for r in redundant_files[:200]: # Show first 200
    gphotos_link = f"https://photos.google.com/search/{r['date']}"
    html_content += f"""
      <tr>
        <td>{r['date']}</td>
        <td><code>{r['filename']}</code></td>
        <td>{r['size_mb']:.2f} MB</td>
        <td><span class='badge'>Redundant Burst</span></td>
        <td><a class='btn' href='{gphotos_link}' target='_blank'>Find in Google Photos ↗</a></td>
      </tr>
    """

html_content += """
    </tbody>
  </table>
</body>
</html>
"""

with open('/content/photos_cleanup_report.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("Report saved to /content/photos_cleanup_report.html and /content/redundant_photos_to_delete.csv")
display(HTML("<p style='color:green;font-weight:bold;'>✓ Analysis complete! You can download 'photos_cleanup_report.html' and 'redundant_photos_to_delete.csv' from the Colab file tree on the left.</p>"))

### Step 7: (Optional) Auto-Downscale High-Res Photos to 16 MP (WebP/JPEG)
If you want to compress the kept photos so you can re-upload them in Storage Saver quality:

In [ ]:
OPTIMIZED_DIR = '/content/optimized_photos'
os.makedirs(OPTIMIZED_DIR, exist_ok=True)

# Filter to keep only the best shots from clusters + non-clustered photos
keep_paths = set(r['path'] for r in records) - set(rf['path'] for rf in redundant_files)
print(f"Compressing {len(keep_paths)} photos to Google Storage Saver WebP...")

saved_bytes = 0
for p in tqdm(list(keep_paths)[:50]): # Example on first 50
    try:
        orig_size = os.path.getsize(p)
        with Image.open(p) as img:
            # Resize if > 16 MP (max dimension ~ 4600px)
            w, h = img.size
            if w > 4600 or h > 4600:
                img.thumbnail((4600, 4600), Image.Resampling.LANCZOS)
            
            out_filename = Path(p).stem + '.webp'
            out_path = os.path.join(OPTIMIZED_DIR, out_filename)
            img.save(out_path, 'WEBP', quality=82, method=6)
            saved_bytes += (orig_size - os.path.getsize(out_path))
    except Exception:
        continue

print(f"Downscale complete! Reclaimed {saved_bytes / (1024*1024):.1f} MB during sample compression.")